# 01 - Train a custom baseline (U-Net / MedNeXt / DA-VNet / AKS-UNet)

Runtime -> Change runtime type -> **T4 GPU**. Each team member sets `MODEL` to their model.
Training checkpoints to Drive every epoch; if Colab disconnects or units run out, **just re-run all cells** and it
resumes. Set `MAX_HOURS` a little below your remaining session so it stops cleanly.

In [1]:

MODEL = 'unet3d'
SMOKE = False
EPOCHS = 150
MAX_HOURS = 8
PATCH = '192 192 16'
BATCH = 2


In [3]:
import os

DATA_ROOT = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo"

CACHE = os.path.join(DATA_ROOT, "cache_0.9mm")
CLEAN = os.path.join(DATA_ROOT, "clean")
SPLITS = os.path.join(DATA_ROOT, "splits.json")

print("Cache:", os.path.exists(CACHE))
print("Clean:", os.path.exists(CLEAN))
print("Splits:", os.path.isfile(SPLITS))
print("Cache files:", len(os.listdir(CACHE)) if os.path.isdir(CACHE) else 0)
print("Clean files:", len(os.listdir(CLEAN)) if os.path.isdir(CLEAN) else 0)

Cache: True
Clean: True
Splits: True
Cache files: 300
Clean files: 600


In [4]:

# --- Kaggle setup: dataset + GitHub repository ---

import os
import subprocess

# Kaggle dataset (read-only)
DATA_ROOT = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo"

CACHE = os.path.join(DATA_ROOT, "cache_0.9mm")
CLEAN = os.path.join(DATA_ROOT, "clean")
SPLITS = os.path.join(DATA_ROOT, "splits.json")

# Writable project folder for training results
DRIVE = "/kaggle/working/figo"

# GitHub repository
REPO_URL = "https://github.com/acm-research-f26/Radii.git"
REPO = "/kaggle/working/figo-repo"

os.makedirs(DRIVE, exist_ok=True)

# Clone repository or update existing copy
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)

# Switch to repository
%cd {REPO}

# Install dependencies
!pip -q install -r requirements.txt

print("Kaggle setup complete!")
print("Dataset:", DATA_ROOT)
print("Training outputs:", DRIVE)


/kaggle/working/figo-repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 10.1 MB/s eta 0:00:00a 0:00:01
Kaggle setup complete!
Dataset: /kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo
Training outputs: /kaggle/working/figo


In [15]:

# --- Week 2: Full training configuration ---

MODEL = "unet3d"
SMOKE = False
EPOCHS = 150
MAX_HOURS = 8
PATCH = "192 192 16"
BATCH = 2

CACHE = DATA_ROOT + "/cache_0.9mm"
SPLITS = DATA_ROOT + "/splits.json"

# Separate folder for full training
RUN = f"{DRIVE}/runs/full_{MODEL}"

os.makedirs(RUN, exist_ok=True)

print("Model:", MODEL)
print("Epochs:", EPOCHS)
print("Run folder:", RUN)
print("Cache exists:", os.path.isdir(CACHE))
print("Splits exists:", os.path.isfile(SPLITS))


name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Model: aksunet
Cache: /kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/cache_0.9mm
Splits: /kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/splits.json
Run folder: /kaggle/working/figo/runs/smoke_aksunet
Epochs: 2


In [16]:

!python -m figo.train \
    --model {MODEL} \
    --cache-dir {CACHE} \
    --splits {SPLITS} \
    --out-dir {RUN} \
    --epochs {EPOCHS} \
    --max-hours {MAX_HOURS} \
    --patch-size {PATCH} \
    --batch-size {BATCH}


Loading 210 training volumes ...
aksunet: 21.2M parameters, deep supervision=True
epoch=0 lr=1.00e-02 train_loss=1.3248 epoch_sec=112
epoch=1 lr=5.36e-03 train_loss=1.0889 val_wall=0.3241 val_cavity=0.0000 val_myoma=0.0000 val_cyst=0.0000 val_mean=0.0810 val_ema=0.0810 epoch_sec=142
Done. Best smoothed val Dice 0.0810


In [ ]:
# timing -> how many epochs fit the budget (row 1 = pure training, row 2 = training + validation)
import csv
log = list(csv.DictReader(open(f'{RUN}/log.csv')))
for r in log:
    print('epoch', r['epoch'], r['epoch_sec'], 's')
if len(log) >= 3:
    t_train = float(log[1]['epoch_sec'])
    t_val = max(float(log[2]['epoch_sec']) - t_train, 0)
    per_epoch = t_train + t_val / 10          # validation runs every 10 epochs
    print(f'~{per_epoch:.0f} s per epoch on average; validation ~{t_val:.0f} s')
    for h in (6, 8, 10):
        print(f'  {h:2d} GPU-hours  ->  EPOCHS = {int(h * 3600 / per_epoch)}')
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# learning curves
import csv, matplotlib.pyplot as plt
log = list(csv.DictReader(open(f'{RUN}/log.csv')))
plt.plot([float(r['train_loss']) for r in log], label='train loss')
v = [(int(r['epoch']), float(r['val_mean'])) for r in log if r['val_mean']]
if v: plt.plot(*zip(*v), 'o-', label='val mean Dice')
plt.legend(); plt.xlabel('epoch'); plt.show()

## Validation-set inference and evaluation (Week 2)

In [8]:
%pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import subprocess

REPO = "/kaggle/working/figo-repo"
REPO_URL = "https://github.com/acm-research-f26/Radii.git"

if not os.path.isdir(REPO):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO],
        check=True
    )

print("Repository ready:", os.path.isdir(REPO))
print("FIGO code found:", os.path.isfile(f"{REPO}/figo/infer.py"))

Cloning into '/kaggle/working/figo-repo'...


Repository ready: True
FIGO code found: True


In [ ]:
import os

best = f"{RUN}/best.pt"

if not os.path.isfile(best):
    raise FileNotFoundError(
        f"No trained checkpoint found: {best}"
    )

print("Checkpoint found:", best)
print("Size (MB):", round(os.path.getsize(best) / 1024**2, 2))

In [ ]:

# --- Week 2: Validation inference and evaluation ---

PRED = f"{DRIVE}/preds/{MODEL}"
RES = f"{DRIVE}/results_val"

os.makedirs(PRED, exist_ok=True)
os.makedirs(RES, exist_ok=True)

!python -m figo.infer \
    --checkpoint {RUN}/best.pt \
    --data-dir {CLEAN} \
    --splits {SPLITS} \
    --subset val \
    --out-dir {PRED}

!python -m figo.evaluate \
    --pred-dir {PRED} \
    --data-dir {CLEAN} \
    --splits {SPLITS} \
    --subset val \
    --out {RES}/{MODEL}_seg.csv

!python -m figo.figo_eval agreement \
    --ref-dir {CLEAN} \
    --pred-dir {PRED} \
    --splits {SPLITS} \
    --subset val \
    --out {RES}/{MODEL}_figo_agreement.csv


In [12]:
!nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv

index, name, memory.used [MiB], memory.total [MiB]
0, Tesla T4, 0 MiB, 15360 MiB
1, Tesla T4, 0 MiB, 15360 MiB


In [2]:
import os

DATA_ROOT = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo"
CLEAN = os.path.join(DATA_ROOT, "clean")

print("First 10 files in clean:")
print(sorted(os.listdir(CLEAN))[:10])

print("\nFiles containing 221129_014:")
print([f for f in os.listdir(CLEAN) if "221129_014" in f])

First 10 files in clean:
['UMD_221129_001_seg.nii', 'UMD_221129_001_t2.nii', 'UMD_221129_002_seg.nii', 'UMD_221129_002_t2.nii', 'UMD_221129_003_seg.nii', 'UMD_221129_003_t2.nii', 'UMD_221129_004_seg.nii', 'UMD_221129_004_t2.nii', 'UMD_221129_005_seg.nii', 'UMD_221129_005_t2.nii']

Files containing 221129_014:
['UMD_221129_014_seg.nii', 'UMD_221129_014_t2.nii']


In [3]:
import os
import gzip
import shutil

CLEAN_GZ = "/kaggle/working/figo/data/clean"
os.makedirs(CLEAN_GZ, exist_ok=True)

files = [f for f in os.listdir(CLEAN) if f.endswith(".nii")]

for filename in files:
    source = os.path.join(CLEAN, filename)
    destination = os.path.join(CLEAN_GZ, filename + ".gz")

    if not os.path.exists(destination):
        with open(source, "rb") as src:
            with gzip.open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)

print("Compressed files:", len(os.listdir(CLEAN_GZ)))
print("Example:", sorted(os.listdir(CLEAN_GZ))[:2])

Compressed files: 600
Example: ['UMD_221129_001_seg.nii.gz', 'UMD_221129_001_t2.nii.gz']


In [4]:
import os

print("Saved checkpoint locations:")

for root, dirs, files in os.walk("/kaggle/input"):
    if "best.pt" in files:
        print(os.path.join(root, "best.pt"))

Saved checkpoint locations:
/kaggle/input/notebooks/snigdhahy3/figo-training/figo/runs/full_unet3d/best.pt


In [9]:
import os
import subprocess

CHECKPOINT = "/kaggle/input/notebooks/snigdhahy3/figo-training/figo/runs/full_unet3d/best.pt"
CLEAN_GZ = "/kaggle/working/figo/data/clean"
SPLITS = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/splits.json"
PRED = "/kaggle/working/figo/preds/unet3d"

os.makedirs(PRED, exist_ok=True)

subprocess.run([
    "python", "-m", "figo.infer",
    "--checkpoint", CHECKPOINT,
    "--data-dir", CLEAN_GZ,
    "--splits", SPLITS,
    "--subset", "val",
    "--out-dir", PRED
], cwd="/kaggle/working/figo-repo", check=True)

print("U-Net validation inference completed!")

[1/30] UMD_221129_014
[2/30] UMD_221129_016
[3/30] UMD_221129_020
[4/30] UMD_221129_045
[5/30] UMD_221129_066
[6/30] UMD_221129_074
[7/30] UMD_221129_077
[8/30] UMD_221129_092
[9/30] UMD_221129_093
[10/30] UMD_221129_131
[11/30] UMD_221129_141
[12/30] UMD_221129_147
[13/30] UMD_221129_153
[14/30] UMD_221129_164
[15/30] UMD_221129_173
[16/30] UMD_221129_183
[17/30] UMD_221129_204
[18/30] UMD_221129_205
[19/30] UMD_221129_206
[20/30] UMD_221129_210
[21/30] UMD_221129_228
[22/30] UMD_221129_231
[23/30] UMD_221129_245
[24/30] UMD_221129_250
[25/30] UMD_221129_263
[26/30] UMD_221129_266
[27/30] UMD_221129_273
[28/30] UMD_221129_281
[29/30] UMD_221129_289
[30/30] UMD_221129_296
U-Net validation inference completed!


In [2]:
import os
import subprocess

PRED = "/kaggle/working/figo/preds/unet3d"
CLEAN_GZ = "/kaggle/working/figo/data/clean"
SPLITS = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/splits.json"
RES = "/kaggle/working/figo/results_val"

os.makedirs(RES, exist_ok=True)

print("Predictions found:", len([
    f for f in os.listdir(PRED) if f.endswith(".nii.gz")
]))

subprocess.run([
    "python", "-m", "figo.evaluate",
    "--pred-dir", PRED,
    "--data-dir", CLEAN_GZ,
    "--splits", SPLITS,
    "--subset", "val",
    "--out", f"{RES}/unet3d_seg.csv"
], cwd="/kaggle/working/figo-repo", check=True)

print("U-Net validation Dice evaluation completed!")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/figo/preds/unet3d'

In [3]:
import os

CHECKPOINT = "/kaggle/input/notebooks/snigdhahy3/figo-training/figo/runs/full_unet3d/best.pt"

print("Checkpoint exists:", os.path.exists(CHECKPOINT))

Checkpoint exists: True


In [4]:
import os
import subprocess

REPO = "/kaggle/working/figo-repo"

if not os.path.exists(REPO):
    subprocess.run([
        "git", "clone",
        "https://github.com/acm-research-f26/Radii.git",
        REPO
    ], check=True)

print("Repository ready:", os.path.exists(REPO))

Cloning into '/kaggle/working/figo-repo'...


Repository ready: True


In [5]:
import os
import gzip
import shutil

CLEAN = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/clean"
CLEAN_GZ = "/kaggle/working/figo/data/clean"

os.makedirs(CLEAN_GZ, exist_ok=True)

files = [f for f in os.listdir(CLEAN) if f.endswith(".nii")]

for filename in files:
    source = os.path.join(CLEAN, filename)
    destination = os.path.join(CLEAN_GZ, filename + ".gz")

    if not os.path.exists(destination):
        with open(source, "rb") as src:
            with gzip.open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)

print("Compressed files:", len(os.listdir(CLEAN_GZ)))

Compressed files: 600


In [6]:
%pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.4 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import subprocess

CHECKPOINT = "/kaggle/input/notebooks/snigdhahy3/figo-training/figo/runs/full_unet3d/best.pt"
CLEAN_GZ = "/kaggle/working/figo/data/clean"
SPLITS = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/splits.json"
PRED = "/kaggle/working/figo/preds/unet3d"

os.makedirs(PRED, exist_ok=True)

subprocess.run([
    "python", "-m", "figo.infer",
    "--checkpoint", CHECKPOINT,
    "--data-dir", CLEAN_GZ,
    "--splits", SPLITS,
    "--subset", "val",
    "--out-dir", PRED
], cwd="/kaggle/working/figo-repo", check=True)

print("U-Net validation inference completed!")

[1/30] UMD_221129_014
[2/30] UMD_221129_016
[3/30] UMD_221129_020
[4/30] UMD_221129_045
[5/30] UMD_221129_066
[6/30] UMD_221129_074
[7/30] UMD_221129_077
[8/30] UMD_221129_092
[9/30] UMD_221129_093
[10/30] UMD_221129_131
[11/30] UMD_221129_141
[12/30] UMD_221129_147
[13/30] UMD_221129_153
[14/30] UMD_221129_164
[15/30] UMD_221129_173
[16/30] UMD_221129_183
[17/30] UMD_221129_204
[18/30] UMD_221129_205
[19/30] UMD_221129_206
[20/30] UMD_221129_210
[21/30] UMD_221129_228
[22/30] UMD_221129_231
[23/30] UMD_221129_245
[24/30] UMD_221129_250
[25/30] UMD_221129_263
[26/30] UMD_221129_266
[27/30] UMD_221129_273
[28/30] UMD_221129_281
[29/30] UMD_221129_289
[30/30] UMD_221129_296
U-Net validation inference completed!


In [8]:
import os
import subprocess

RES = "/kaggle/working/figo/results_val"
os.makedirs(RES, exist_ok=True)

subprocess.run([
    "python", "-m", "figo.evaluate",
    "--pred-dir", PRED,
    "--data-dir", CLEAN_GZ,
    "--splits", SPLITS,
    "--subset", "val",
    "--out", f"{RES}/unet3d_seg.csv"
], cwd="/kaggle/working/figo-repo", check=True)

print("U-Net Dice evaluation completed!")
print("Results saved:", os.path.exists(f"{RES}/unet3d_seg.csv"))

  10/30
  20/30
  30/30

30 cases from /kaggle/working/figo/preds/unet3d
  dice_wall                           0.821 +/- 0.120
  hd95_wall                          16.858 +/- 36.600
  assd_wall                           4.525 +/- 12.703
  dice_cavity                         0.689 +/- 0.228
  hd95_cavity                         6.531 +/- 11.568  (1 missed)
  assd_cavity                         1.530 +/- 3.031  (1 missed)
  dice_myoma                          0.525 +/- 0.347
  hd95_myoma                         33.077 +/- 53.024  (1 missed)
  assd_myoma                         14.881 +/- 30.790  (1 missed)
  dice_cyst                           0.000 +/- 0.000
  hd95_cavity_wall_interface          6.515 +/- 11.641  (1 missed)
  assd_cavity_wall_interface          1.531 +/- 3.078  (1 missed)
  hd95_serosa                        20.133 +/- 40.096
  assd_serosa                         6.950 +/- 20.750
  dice_uterus                         0.851 +/- 0.191
  macro_dice_wall_cavity_myoma       

In [9]:
import os
import subprocess

RES = "/kaggle/working/figo/results_val"

subprocess.run([
    "python", "-m", "figo.figo_eval", "agreement",
    "--ref-dir", CLEAN_GZ,
    "--pred-dir", PRED,
    "--splits", SPLITS,
    "--subset", "val",
    "--out", f"{RES}/unet3d_figo_agreement.csv"
], cwd="/kaggle/working/figo-repo", check=True)

print("FIGO agreement evaluation completed!")
print("Results saved:", os.path.exists(
    f"{RES}/unet3d_figo_agreement.csv"
))

  n_ref_fibroids         111
  n_pred_fibroids        105
  n_matched              60
  detection_recall       0.541
  detection_precision    0.571
  exact_agreement        0.700
  coarse_agreement       0.850
  kappa_exact            0.499
  kappa_coarse           0.685
FIGO agreement evaluation completed!
Results saved: True


In [10]:
import os
import shutil

BASE = "/kaggle/working/figo"

pred_zip = shutil.make_archive(
    "/kaggle/working/unet3d_predictions",
    "zip",
    root_dir=BASE,
    base_dir="preds"
)

results_zip = shutil.make_archive(
    "/kaggle/working/unet3d_results_val",
    "zip",
    root_dir=BASE,
    base_dir="results_val"
)

for path in [pred_zip, results_zip]:
    print(
        os.path.basename(path),
        "—",
        round(os.path.getsize(path) / 1024**2, 2),
        "MB"
    )

print("Both ZIP files created successfully!")

unet3d_predictions.zip — 0.82 MB
unet3d_results_val.zip — 0.01 MB
Both ZIP files created successfully!


In [11]:
from IPython.display import display, FileLink
import os

files = [
    "/kaggle/working/unet3d_predictions.zip",
    "/kaggle/working/unet3d_results_val.zip"
]

for file in files:
    print(os.path.basename(file), "exists:", os.path.exists(file))
    if os.path.exists(file):
        display(FileLink(file))

unet3d_predictions.zip exists: True


/kaggle/working/unet3d_predictions.zip

unet3d_results_val.zip exists: True


/kaggle/working/unet3d_results_val.zip